# AI-Based Fake News Detection System
**Student:** Sumanth 
**Department:** School of Computer Science and Applications  

---
This notebook covers the complete machine learning pipeline:
- Data preparation
- Text preprocessing
- Feature extraction (TF-IDF)
- Model training & evaluation
- Confusion matrix & classification report
- Model comparison visualization

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_curve, auc
)
from sklearn.pipeline import Pipeline

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#0a0a0f',
    'axes.facecolor': '#13131e',
    'axes.edgecolor': '#2a2a3a',
    'text.color': '#f0f0f8',
    'axes.labelcolor': '#f0f0f8',
    'xtick.color': '#888',
    'ytick.color': '#888',
    'grid.color': '#1a1a2a',
    'font.family': 'monospace'
})

print('✅ Libraries imported successfully')

## 2. Dataset Preparation

In [ ]:
REAL_NEWS = [
    'Scientists at NASA confirmed the launch of the Artemis mission scheduled for next year, citing successful tests.',
    'The Federal Reserve raised interest rates by 25 basis points in response to ongoing inflation pressures.',
    'Researchers at Johns Hopkins University published a peer-reviewed study on the efficacy of new cancer treatments.',
    'The United Nations Security Council convened an emergency session to address the escalating humanitarian crisis.',
    'Apple reported quarterly earnings that exceeded analyst expectations, with iPhone sales up 8 percent year over year.',
    'Climate scientists warn that global temperatures are on track to exceed the 1.5 degree Celsius threshold by 2035.',
    'The World Health Organization approved a new malaria vaccine showing 77 percent efficacy in clinical trials.',
    'Government officials released the annual budget report showing a modest decrease in national deficit figures.',
    'The Supreme Court ruled in a 6-3 decision on the landmark case concerning digital privacy rights.',
    'Economists predict moderate GDP growth of 2.3 percent for the upcoming fiscal year based on current trends.',
    'International trade negotiations concluded with a new agreement expected to boost exports significantly.',
    'Medical experts recommend updated COVID-19 booster shots ahead of the winter season.',
    'Tech giant Google announced layoffs affecting 12000 employees as part of restructuring efforts.',
    'The European Central Bank maintained its key interest rate amid concerns about economic slowdown.',
    'New archaeological findings in Egypt shed light on previously unknown pharaonic dynasty.',
    'A new electric vehicle battery technology promises to double current range on a single charge.',
    'Congressional leaders reached a bipartisan deal on infrastructure spending worth 1.2 trillion dollars.',
    'The International Monetary Fund revised its global growth forecast downward due to geopolitical tensions.',
    'Scientists successfully tested a quantum computing processor achieving record-breaking speeds.',
    'Public health officials confirmed the outbreak of a new flu strain and urge vaccination.',
    'India launched its first crewed lunar mission from the Satish Dhawan Space Centre in Sriharikota.',
    'The Reserve Bank of India held repo rate steady at 6.5 percent citing controlled inflation.',
    'ISRO successfully deployed the communication satellite GSAT-20 into geostationary orbit.',
    'Union Budget 2024 allocated increased funds toward education and rural development sectors.',
    "India's GDP grew at 7.2 percent in the last quarter, reaffirming its position as fastest growing economy.",
    'The Indian government launched PM Vishwakarma scheme to support traditional artisans nationwide.',
    'IIT researchers developed a low-cost water purification system suitable for rural communities.',
    'WHO designated new health emergency protocols following outbreak reports from Southeast Asia.',
    'SpaceX Starship completed its fourth test flight successfully, splashing down in the Indian Ocean.',
    'Microsoft announced integration of AI features across its Office 365 productivity suite.',
    'A study published in Nature found microplastics in 90 percent of tested ocean samples worldwide.',
    'Bengaluru metro expanded its network with three new stations opening on the Purple Line extension.',
    'The Paris Agreement signatories met to review progress on carbon emission reduction commitments.',
    'Global chip shortage begins to ease as TSMC increases semiconductor manufacturing capacity.',
    'New research links excessive social media use with increased anxiety among teenagers.',
    'Scientists discover a new species of deep sea fish near hydrothermal vents in Pacific Ocean.',
]

FAKE_NEWS = [
    'BREAKING: Government secretly puts microchips in vaccines to track citizens worldwide confirmed by insider.',
    'Scientists HIDE proof that the Earth is flat and NASA fakes all space missions with CGI technology.',
    'Miracle cure discovered: drinking bleach every morning eliminates all diseases doctors wont tell you.',
    'SHOCKING: Bill Gates admits he engineered COVID-19 to reduce world population by 90 percent.',
    '5G towers cause cancer and spread coronavirus this is what big telecom doesnt want you to know.',
    'Deep state elites drinking child blood to stay young exposed by anonymous whistleblower source.',
    'Chemtrails confirmed as mind control chemicals sprayed by government to make people obedient.',
    'Pope Francis secretly converts to Islam and announces end of Catholic Church next month.',
    'Hollywood celebrities are actually lizard people in disguise revealed by former NASA employee.',
    'Drinking urine cures cancer doctors and pharmaceutical companies suppress this natural remedy.',
    'URGENT: New world order plans to microchip all humans by 2025 resist the vaccine agenda now.',
    'Elvis Presley still alive spotted working at grocery store in Tennessee exclusive photos inside.',
    'Moon landing was completely faked Stanley Kubrick admitted it on his deathbed according to sources.',
    'Secret society controls all world governments and plans to eliminate 80 percent of population.',
    'Eating magnets gives you superpowers this is what the government does not want you to discover.',
    'EXPOSED: Dinosaurs never existed they were invented by scientists to disprove religion evidence found.',
    'WiFi radiation causes brain tumors in children tech companies pay doctors to hide the truth.',
    'Ancient aliens built the pyramids and are returning in 2024 to reclaim earth from humans.',
    'Sunscreen causes skin cancer and is a plot by pharmaceutical companies to keep you unhealthy.',
    'Obama born in Kenya new documents prove citizenship was faked insider leaks bombshell evidence.',
    'CONFIRMED: Fluoride in water is a government mind control program to make people docile and stupid.',
    'Famous actor dies in secret celebrity illuminati sacrifice mainstream media covers it up.',
    'Quantum physics proves heaven exists scientists discover parallel universe where dead people live.',
    'GMO foods contain hidden DNA altering chemicals to sterilize the population expose the agenda.',
    'Astrologer predicts massive earthquake will destroy California within 48 hours flee immediately.',
    'Natural doctors curing cancer with herbs being killed by big pharma to protect billion dollar industry.',
    'BOMBSHELL: Indian PM secretly working for foreign powers to sell national secrets leaked documents.',
    'Time travel machine invented in secret government lab photos leaked show future cities.',
    'Meditation app secretly records your thoughts using brain wave technology and sells to advertisers.',
    'Eating raw garlic every hour cures coronavirus doctors are being silenced for speaking the truth.',
    'Major Indian bank about to collapse government hiding financial crisis protect your savings now.',
    'Secret tunnel discovered under White House leading to underground city for elite survival.',
    'Chocolate milk comes from brown cows this fact has been suppressed by dairy industry for decades.',
    'Aliens have landed in Antarctica global governments covering up first contact since December.',
    'New phone update downloads your bank account information do not update your phone.',
    'Politician photographed with devil worshippers at secret ceremony exclusive footage surfaces.',
]

real_df = pd.DataFrame({'text': REAL_NEWS, 'label': 0, 'category': 'Real'})
fake_df = pd.DataFrame({'text': FAKE_NEWS, 'label': 1, 'category': 'Fake'})
df = pd.concat([real_df, fake_df], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
print(f'Class distribution:\n{df["category"].value_counts()}')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
df['word_count'] = df['text'].apply(lambda x: len(x.split()))
df['char_count'] = df['text'].apply(len)
df['exclamation'] = df['text'].apply(lambda x: x.count('!'))
df['caps_ratio'] = df['text'].apply(lambda x: sum(1 for c in x if c.isupper()) / max(len(x), 1))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Exploratory Data Analysis — Fake vs Real News', fontsize=14, color='#7fffb2', y=1.01)

colors = {'Real': '#7fffb2', 'Fake': '#ff6b6b'}

# Word count distribution
for cat, grp in df.groupby('category'):
    axes[0,0].hist(grp['word_count'], bins=15, alpha=0.7, label=cat, color=colors[cat], edgecolor='none')
axes[0,0].set_title('Word Count Distribution', color='#f0f0f8')
axes[0,0].set_xlabel('Word Count'); axes[0,0].set_ylabel('Frequency')
axes[0,0].legend()

# Class distribution
counts = df['category'].value_counts()
axes[0,1].bar(counts.index, counts.values, color=['#7fffb2','#ff6b6b'], edgecolor='none', width=0.5)
axes[0,1].set_title('Class Distribution', color='#f0f0f8')
axes[0,1].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0,1].text(i, v + 0.3, str(v), ha='center', color='#f0f0f8', fontsize=11)

# Caps ratio
for cat, grp in df.groupby('category'):
    axes[1,0].hist(grp['caps_ratio'], bins=15, alpha=0.7, label=cat, color=colors[cat], edgecolor='none')
axes[1,0].set_title('Uppercase Ratio Distribution', color='#f0f0f8')
axes[1,0].set_xlabel('Ratio of Uppercase Characters')
axes[1,0].legend()

# Exclamation marks
excl = df.groupby('category')['exclamation'].mean()
axes[1,1].bar(excl.index, excl.values, color=['#7fffb2','#ff6b6b'], edgecolor='none', width=0.5)
axes[1,1].set_title('Avg Exclamation Marks', color='#f0f0f8')
axes[1,1].set_ylabel('Average Count')

plt.tight_layout()
plt.savefig('../docs/eda_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()
print('✅ EDA plot saved')

## 4. Text Preprocessing

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)

# Show before/after
print('BEFORE:', df['text'].iloc[0])
print('AFTER: ', df['clean_text'].iloc[0])

## 5. Train-Test Split & Feature Extraction

In [ ]:
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f'Training samples : {len(X_train)}')
print(f'Testing samples  : {len(X_test)}')
print(f'Train class dist :\n{y_train.value_counts().rename({0:"Real",1:"Fake"})}')

## 6. Model Training & Evaluation

In [ ]:
models = {
    'Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=10000, sublinear_tf=True)),
        ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000)),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
    ]),
    'Naive Bayes': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,1), max_features=5000)),
        ('clf', MultinomialNB(alpha=0.1))
    ]),
}

results = {}
trained = {}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    acc = accuracy_score(y_test, preds)
    results[name] = acc
    trained[name] = pipeline
    print(f'\n{'='*50}')
    print(f'Model: {name}  |  Accuracy: {acc:.4f}')
    print('='*50)
    print(classification_report(y_test, preds, target_names=['Real','Fake']))

best = max(results, key=results.get)
print(f'\n✅ Best model: {best} ({results[best]*100:.2f}%)')

## 7. Confusion Matrix Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Confusion Matrices — All Models', fontsize=13, color='#7fffb2')

for ax, (name, pipeline) in zip(axes, trained.items()):
    preds = pipeline.predict(X_test)
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(
        cm, annot=True, fmt='d', ax=ax,
        cmap='Greens', linewidths=0.5,
        xticklabels=['Real','Fake'],
        yticklabels=['Real','Fake'],
        annot_kws={'size': 14, 'color': 'white'}
    )
    ax.set_title(f'{name}\n({results[name]*100:.1f}%)', color='#f0f0f8')
    ax.set_xlabel('Predicted', color='#f0f0f8')
    ax.set_ylabel('Actual', color='#f0f0f8')

plt.tight_layout()
plt.savefig('../docs/confusion_matrices.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()
print('✅ Confusion matrix plot saved')

## 8. Model Accuracy Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
names = list(results.keys())
accs  = [results[n] * 100 for n in names]
bar_colors = ['#7fffb2' if n == best else '#74b9ff' for n in names]

bars = ax.barh(names, accs, color=bar_colors, edgecolor='none', height=0.4)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_width() - 2, bar.get_y() + bar.get_height()/2,
            f'{acc:.1f}%', va='center', ha='right', color='#0a0a0f', fontweight='bold')

ax.set_xlim(60, 105)
ax.set_xlabel('Accuracy (%)', color='#f0f0f8')
ax.set_title('Model Accuracy Comparison', color='#7fffb2', fontsize=13)
ax.axvline(x=90, color='#ffd166', linestyle='--', alpha=0.5, linewidth=1)
ax.text(90.5, -0.5, '90% threshold', color='#ffd166', fontsize=8)

green = mpatches.Patch(color='#7fffb2', label='Best Model')
blue  = mpatches.Patch(color='#74b9ff', label='Other Models')
ax.legend(handles=[green, blue])

plt.tight_layout()
plt.savefig('../docs/model_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()
print('✅ Model comparison plot saved')

## 9. Live Prediction Demo

In [ ]:
best_model = trained[best]

test_cases = [
    'ISRO successfully launched a new weather satellite from Sriharikota last Tuesday.',
    'SHOCKING: Government hiding alien bodies in secret facility doctors confirm.',
    "India's economy grew at 7.2 percent last quarter according to official data.",
    'Miracle cure for cancer found: eat this one fruit every morning doctors hate it.',
    'The Supreme Court issued a landmark ruling on data privacy rights today.',
]

print(f'{'='*60}')
print(f'LIVE PREDICTION DEMO — Using: {best}')
print(f'{'='*60}')

for i, text in enumerate(test_cases, 1):
    cleaned = clean_text(text)
    pred    = best_model.predict([cleaned])[0]
    prob    = best_model.predict_proba([cleaned])[0]
    label   = 'FAKE' if pred == 1 else 'REAL'
    conf    = max(prob) * 100
    icon    = '⚠' if pred == 1 else '✓'
    print(f'\n[{i}] {text[:70]}...' if len(text) > 70 else f'\n[{i}] {text}')
    print(f'    {icon} {label}  |  Confidence: {conf:.1f}%  |  Real: {prob[0]*100:.1f}%  Fake: {prob[1]*100:.1f}%')

## 10. Summary

| Model | Accuracy |
|---|---|
| Logistic Regression | **Best** |
| Naive Bayes | High |
| Random Forest | Good |

**Conclusion:** The Logistic Regression model with TF-IDF n-gram (1,2) features achieved the best performance on the test set. The system successfully distinguishes real news from fake news using linguistic patterns, keyword analysis, and statistical text features.

**Future Scope:**
- Train on larger labelled datasets (LIAR, FakeNewsNet)
- Integrate transformer models (BERT, RoBERTa)
- Add URL and source credibility checking
- Browser extension integration